# Eval-determinism gate — does LIBERO's eval path give arm-matched episodes?

**This is the gate for the winning paper.** It answers one question, on CPU, with no checkpoint:

> When two policy arms take a different number of steps, do they see the same evaluation
> episodes, or different ones?

If they see different ones, **every paired comparison published on the stock LIBERO eval path is
confounded** — including the A_clean vs B_dim comparison behind our own +18pp result.

## Why this supersedes the 2026-08-25 check

`mhh-gate/run_gate.py::verify_eval_determinism` ran one eval shard twice and compared per-episode
length and success vectors. It returned `IDENTICAL:False` on 2026-08-25. That result was never
usable, because it had **two confounds baked in**:

1. **Mismatched episode counts** — run A produced 6 episodes, run B produced 5 (unseeded autoreset
   extras / invalid-episode filtering). `la == lb` on unequal-length lists is False for free.
2. **Unpinned policy sampling** — GR00T's action head samples. Identical observations still yield
   different actions, hence different trajectory lengths, with no environment nondeterminism at all.

**This notebook removes both by construction rather than by re-running and hoping:**

- Episode count is **fixed explicitly**; nothing is filtered.
- **There is no policy.** Actions come from a pinned `np.random.RandomState`. Any difference in
  the environment's state is therefore the environment.

That is also why it needs **no checkpoint and no GPU** — the question is a property of
LIBERO/robosuite/Gymnasium, not of the trained model.

## The mechanism being tested, stated before the run

Gymnasium's own API documentation, verbatim:

> *"However, if the environment already has a PRNG and **seed=None** is passed, **the PRNG will
> not be reset** and the env's np_random_seed will not be altered."*

And in LIBERO: `LiberoEnv.reset()` calls `self._env.seed(int(seed))` then `self._env.reset()`, never
`set_init_state()`; and only the **first** reset of a rollout is seeded
(`rollout_policy.py:302-309`). So every later episode draws from a PRNG whose position depends on
how many times it was advanced — i.e. on how many steps the policy took.

**Prediction: T2 below fails.** Writing that down first.

## Pre-registered outcomes

| Test | What it checks | Expected |
|---|---|---|
| **T1** | `seed(s)` + `reset()` twice, fresh envs → identical first observation? | PASS |
| **T2** | reset *after* a rollout of K steps, no reseed → does the initial state depend on K? | **FAIL (state depends on K)** |
| **T3** | same as T2 but reseeding explicitly every episode | PASS — this is the fix |

**VERDICT = HARNESS SOUND** only if T1 and T2 both pass. **T2 failing is the paper**, not a bug in
this notebook.


In [ ]:
TASK_SUITE = "libero_object"
TASK_NAME  = "pick_up_the_alphabet_soup_and_place_it_in_the_basket"
SEEDS      = [11, 12, 13]      # three seeds, per the brief
K_SHORT    = 40                # "arm A" takes this many steps before the next reset
K_LONG     = 80                # "arm B" takes this many -- the only difference between arms
IMG_H, IMG_W = 128, 128
MUJOCO_VERSION = "3.3.3"

DRIVE_DIR = "/content/drive/MyDrive/mhh-determinism-gate"
print("seeds", SEEDS, "| K_SHORT", K_SHORT, "| K_LONG", K_LONG)

## 1 · Drive + install

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os; os.makedirs(DRIVE_DIR, exist_ok=True)

import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q",
                "git+https://github.com/Lifelong-Robot-Learning/LIBERO.git"], check=True)
subprocess.run([sys.executable,"-m","pip","install","-q",f"mujoco=={MUJOCO_VERSION}"], check=True)

import mujoco
assert mujoco.__version__ == MUJOCO_VERSION, f"pin failed: {mujoco.__version__}"
print("mujoco", mujoco.__version__)

freeze = subprocess.run([sys.executable,"-m","pip","freeze"],capture_output=True,text=True).stdout
open(f"{DRIVE_DIR}/freeze.txt","w").write(freeze)
print(len(freeze.splitlines()), "packages pinned to freeze.txt")

## 2 · Helpers

`rollout_then_reset` is the whole experiment: seed once, run `k` steps with a **pinned** action
sequence, then reset **without reseeding** — exactly what LIBERO's eval loop does after its first
episode — and return the observation the next episode starts from.


In [ ]:
import numpy as np, hashlib, json, os

def make_env():
    from libero.libero import benchmark, get_libero_path
    from libero.libero.envs import OffScreenRenderEnv
    b = benchmark.get_benchmark_dict()[TASK_SUITE]()
    tid = [b.get_task(i).name for i in range(b.n_tasks)].index(TASK_NAME)
    t = b.get_task(tid)
    bddl = os.path.join(get_libero_path("bddl_files"), t.problem_folder, t.bddl_file)
    return OffScreenRenderEnv(bddl_file_name=bddl, camera_heights=IMG_H, camera_widths=IMG_W)

def obs_key(obs):
    """Stable fingerprint of an initial state: the rendered frame plus the robot state."""
    parts = []
    for k in sorted(obs.keys()):
        v = obs[k]
        if isinstance(v, np.ndarray):
            parts.append(k.encode()); parts.append(np.ascontiguousarray(v).tobytes())
    return hashlib.sha256(b"".join(parts)).hexdigest()[:16]

def fixed_actions(k, tag):
    """Deterministic action sequence. Pinned per tag so no sampling enters the comparison."""
    rs = np.random.RandomState(abs(hash(tag)) % (2**31))
    return [rs.uniform(-0.2, 0.2, size=7) for _ in range(k)]

def rollout_then_reset(seed, k, tag):
    env = make_env()
    env.seed(seed)
    env.reset()                                  # episode 1: SEEDED
    for a in fixed_actions(k, tag):
        env.step(a)
    obs = env.reset()                            # episode 2: NOT reseeded -- the thing under test
    env.close()
    return obs_key(obs)

def seeded_reset(seed):
    env = make_env(); env.seed(seed); obs = env.reset(); env.close()
    return obs_key(obs)

def reseeded_after_rollout(seed, k, tag):
    env = make_env()
    env.seed(seed); env.reset()
    for a in fixed_actions(k, tag):
        env.step(a)
    env.seed(seed + 1); obs = env.reset()        # the FIX: reseed explicitly every episode
    env.close()
    return obs_key(obs)

print("helpers ready")

## 3 · T1 — is a seeded reset reproducible at all?

If this fails, nothing else in the notebook is interpretable and the problem is the install, not
LIBERO.


In [ ]:
t1 = {}
for s in SEEDS:
    a, b = seeded_reset(s), seeded_reset(s)
    t1[s] = {"a": a, "b": b, "match": a == b}
    print(f"seed {s}: {a} vs {b}  ->  {'MATCH' if a==b else 'DIFFER'}")

T1_PASS = all(v["match"] for v in t1.values())
print("\nT1:", "PASS - seeded resets reproduce" if T1_PASS else "FAIL - not reproducible even when seeded")

## 4 · T2 — the actual question

Two arms, identical seed, identical everything **except how many steps they took**. In a sound
harness the next episode's initial state is the same for both. If it is not, two policies that
happen to differ in episode length are being scored on different episodes.


In [ ]:
t2 = {}
for s in SEEDS:
    short = rollout_then_reset(s, K_SHORT, "armA")
    long_ = rollout_then_reset(s, K_LONG,  "armB")
    ctrl  = rollout_then_reset(s, K_SHORT, "armA")      # control: same K, same actions
    t2[s] = {"k_short": short, "k_long": long_, "control": ctrl,
             "arms_match": short == long_, "control_match": short == ctrl}
    print(f"seed {s}:  K={K_SHORT} -> {short}   K={K_LONG} -> {long_}   "
          f"arms {'MATCH' if short==long_ else 'DIFFER'} | control "
          f"{'ok' if short==ctrl else 'UNSTABLE'}")

CONTROL_OK = all(v["control_match"] for v in t2.values())
T2_PASS    = all(v["arms_match"]    for v in t2.values())
print()
print("control:", "ok - repeats of the same arm agree" if CONTROL_OK
      else "UNSTABLE - repeats of the SAME arm differ; T2 cannot be interpreted")
print("T2:", "PASS - initial state independent of step count" if T2_PASS
      else "FAIL - initial state DEPENDS on how many steps the arm took")

## 5 · T3 — does explicit reseeding fix it?

This is the proposed remedy: seed every episode rather than only the first. If T2 fails and T3
passes, the paper has a finding *and* a one-line fix, which is a much stronger submission than the
finding alone.


In [ ]:
t3 = {}
for s in SEEDS:
    a = reseeded_after_rollout(s, K_SHORT, "armA")
    b = reseeded_after_rollout(s, K_LONG,  "armB")
    t3[s] = {"k_short": a, "k_long": b, "match": a == b}
    print(f"seed {s}: {a} vs {b}  ->  {'MATCH' if a==b else 'DIFFER'}")

T3_PASS = all(v["match"] for v in t3.values())
print()
if T3_PASS:
    print("T3: PASS - explicit per-episode reseeding removes the dependence. This is the fix.")
else:
    print("T3: FAIL - reseeding does not fix it; the cause is elsewhere in the reset path.")


## 6 · Verdict — dated, written to Drive, copy into the vault

In [ ]:
import datetime, csv

ts = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
if not CONTROL_OK:
    verdict = "INCONCLUSIVE"
    reading = ("Repeats of the SAME arm disagree, so the harness is noisy for a reason this test "
               "does not isolate. Do not use T2 either way.")
elif T1_PASS and T2_PASS:
    verdict = "HARNESS SOUND"
    reading = ("Seeded resets reproduce AND the next episode's initial state does not depend on "
               "step count. Paired comparisons on this path are arm-matched. The 2026-08-25 "
               "IDENTICAL:False was the check's two confounds, not the design.")
elif T1_PASS and not T2_PASS:
    verdict = "CONFOUND CONFIRMED"
    reading = ("Seeded resets reproduce, but the next episode's initial state DEPENDS on how many "
               "steps the arm took. Two arms of a paired comparison are scored on different "
               "episodes. This is the paper.")
else:
    verdict = "BROKEN"
    reading = "Seeded resets are not reproducible at all; suspect the install before LIBERO."

out = {"timestamp_utc": ts, "verdict": verdict, "reading": reading,
       "mujoco": MUJOCO_VERSION, "task": TASK_NAME, "seeds": SEEDS,
       "k_short": K_SHORT, "k_long": K_LONG,
       "T1_pass": T1_PASS, "T2_pass": T2_PASS, "T3_pass": T3_PASS,
       "control_ok": CONTROL_OK, "T1": t1, "T2": t2, "T3": t3}

open(f"{DRIVE_DIR}/determinism_gate.json","w").write(json.dumps(out, indent=2, sort_keys=True))
with open(f"{DRIVE_DIR}/determinism_gate.csv","w",newline="") as f:
    w = csv.writer(f); w.writerow(["field","value"])
    for k in ["timestamp_utc","verdict","mujoco","task","T1_pass","T2_pass","T3_pass","control_ok"]:
        w.writerow([k, out[k]])
    for s in SEEDS:
        w.writerow([f"T2_seed{s}_arms_match", t2[s]["arms_match"]])

print("=" * 70)
print(f"VERDICT: {verdict}     ({ts})")
print(reading)
print("=" * 70)
print("wrote determinism_gate.json and .csv to", DRIVE_DIR)
print("\nPaste the verdict line and timestamp into:")
print("  mohanvault/01 Projects/Paper Choice 2026-09-12.md  (section 13)")